# บทเรียน 01 - แนะนำตัวแทน AI

ยินดีต้อนรับสู่บทเรียนแรกในหลักสูตร **AI Agents for Beginners**

**ตัวแทน AI** คือโปรแกรมที่ใช้โมเดลภาษาใหญ่ (LLM) เป็นเครื่องยนต์การวิเคราะห์และสามารถดำเนิน *การกระทำ* ในโลกจริง — เรียกใช้ API, สืบค้นฐานข้อมูล หรือรันโค้ด — เพื่อบรรลุเป้าหมายแทนผู้ใช้

ในสมุดบันทึกนี้คุณจะสร้างตัวแทนแรกของคุณ: **ตัวแทนท่องเที่ยว** ที่แนะนำจุดหมายปลายทางสำหรับวันหยุด ระหว่างทางคุณจะได้เรียนรู้วิธีการ:

1. เชื่อมต่อกับ Azure AI Foundry Agent Service โดยใช้ **Microsoft Agent Framework**
2. มอบ **เครื่องมือ** ให้ตัวแทน — ฟังก์ชัน Python ธรรมดาที่สามารถเรียกใช้ได้
3. รันตัวแทนและตรวจสอบการตอบกลับของมัน
4. สตรีมการตอบกลับของตัวแทนทีละโทเค็น


## Setup

ก่อนที่จะรันโน้ตบุ๊คนี้ ให้แน่ใจว่าคุณได้ทำดังนี้:

1. **มีโปรเจกต์ Azure AI Foundry** ที่มีโมเดลแชทถูกดีพลอยไว้แล้ว (เช่น `gpt-4o-mini`)
2. **เข้าสู่ระบบด้วย Azure CLI** — รันคำสั่ง `az login` ในเทอร์มินัลของคุณ
3. **ตั้งค่าตัวแปรสภาพแวดล้อมที่จำเป็น:**
   - `AZURE_AI_PROJECT_ENDPOINT` — จุดเชื่อมต่อโปรเจกต์ Azure AI Foundry ของคุณ
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ชื่อโมเดลที่คุณได้ดีพลอยไว้

เซลล์ด้านล่างจะติดตั้งแพ็กเกจ Python ที่คุณต้องการ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## การสร้างเอเจนต์ตัวแรกของคุณ

เอเจนต์ต้องการสองสิ่ง:

- **คำแนะนำ** ที่บอกว่า *มันคือใคร* และ *ควรทำตัวอย่างไร* (พรอมต์ระบบ)
- **เครื่องมือ** — ฟังก์ชัน Python ที่ตกแต่งด้วย `@tool` ซึ่งเอเจนต์สามารถเรียกใช้เพื่อดึงข้อมูลหรือดำเนินการต่าง ๆ

ด้านล่างนี้เราจะกำหนดเครื่องมืออย่างง่ายที่คืนรายการจุดหมายปลายทางยอดนิยมสำหรับวันหยุดพักผ่อน เอเจนต์จะใช้เครื่องมือนี้เมื่อผู้ใช้ขอคำแนะนำการท่องเที่ยว


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

สำหรับประสบการณ์ที่โต้ตอบได้มากขึ้น คุณสามารถ **สตรีม** การตอบกลับของเอเจนต์ แทนที่จะรอการตอบกลับเต็มรูปแบบ เอเจนต์จะส่งข้อความเป็นส่วนๆ ขณะที่สร้างข้อความ ซึ่งมีประโยชน์อย่างยิ่งในอินเทอร์เฟซแชทที่คุณต้องการแสดงผลลัพธ์แบบเรียลไทม์


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธี:

- **สร้างโปรไวเดอร์** ที่เชื่อมต่อกับ Azure AI Foundry Agent Service ผ่าน `AzureAIProjectAgentProvider`
- **กำหนดเครื่องมือ** โดยใช้ตัวตกแต่ง `@tool` เพื่อให้เอเจนต์เรียกใช้ฟังก์ชัน Python ของคุณได้
- **เรียกใช้งานเอเจนต์** พร้อมข้อความจากผู้ใช้และพิมพ์ผลลัพธ์ตอบกลับ
- **สตรีมการตอบกลับ** เพื่อแสดงผลแบบเรียลไทม์

ในบทเรียนถัดไปเราจะสำรวจกรอบงาน agentic อย่างละเอียดมากขึ้น และเรียนรู้วิธีเพิ่มเครื่องมือที่ทรงพลังและความสามารถในการคิดวิเคราะห์หลายขั้นตอนให้กับเอเจนต์


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารนี้ได้ถูกแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามให้มีความถูกต้องสูงสุด โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้องได้ เอกสารต้นฉบับในภาษาต้นทางถือเป็นแหล่งข้อมูลที่เชื่อถือได้ที่สุด สำหรับข้อมูลที่สำคัญ ขอแนะนำให้ใช้บริการแปลโดยมืออาชีพที่เป็นมนุษย์ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดที่เกิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
